# Notebook 04 — GradCAM & SHAP: Explainable AI for Satellite Imagery

## Why explainability matters for poverty estimation

Deep learning models trained on satellite imagery can predict socioeconomic indicators
with remarkable accuracy (Jean et al., 2016 *Science*; Yeh et al., 2020 *Nature Communications*).
But **black-box predictions are not enough for policy** — a government or NGO deploying
such a model needs to know *why* a region is predicted to be poor.

This notebook applies two complementary explainability techniques to our EuroSAT ResNet-50:

| Technique | What it shows | Speed |
|-----------|--------------|-------|
| **GradCAM** | Which spatial regions (pixels) most influenced the prediction | Fast — one forward+backward pass |
| **SHAP DeepExplainer** | Which input features (pixels) push prediction up or down vs baseline | Slower — requires background dataset |

### Connection to poverty estimation
In a poverty model, GradCAM heatmaps would reveal whether the model is attending to:
- **Roof materials** (metal/tin vs concrete vs thatch) — a direct proxy for wealth
- **Road density and surface type** — unpaved roads correlate with low income
- **Vegetation** — high NDVI can indicate subsistence farming vs commercial agriculture
- **Nighttime light leakage** — bright pixels around settlements indicate electricity access
- **Building regularity** — informal settlements have irregular, densely-packed structures

This kind of spatial audit is essential for **fairness** — to check the model isn't
using spurious correlations (e.g. cloud shadows, seasonal variation) as poverty proxies.

**Install:** `pip install grad-cam shap`

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from torchvision import transforms

# Add repo root to path so we can import src modules
sys.path.insert(0, str(Path('..').resolve()))
from src.models.resnet import build_resnet50_classifier
from src.data.preprocessing import read_geospatial_rgb, _hwc_to_pil

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load the trained ResNet-50 classifier

In [ ]:
CHECKPOINT = Path('../checkpoints/best_model.pt')
assert CHECKPOINT.exists(), f'Checkpoint not found at {CHECKPOINT}. Run train_on_colab.ipynb first.'

ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
class_names = ckpt['class_names']

model = build_resnet50_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

print(f'Classes ({len(class_names)}): {class_names}')
print(f'Validation macro F1: {ckpt["metrics"]["macro_f1"]:.4f}')

## 2. Collect sample images — 2 per class

In [ ]:
DATA_ROOT = Path('../data/eurosat')
assert DATA_ROOT.exists(), 'EuroSAT data not found. Run the download step first.'

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])

def load_sample(path: Path):
    """Returns (input_tensor CHW, rgb_float32 HWC in [0,1], pil_image)."""
    try:
        hwc = read_geospatial_rgb(path)
        pil = _hwc_to_pil(hwc)
    except Exception:
        pil = Image.open(path).convert('RGB')
        hwc = np.array(pil).astype(np.float32) / 255.0
    pil_resized = pil.resize((224, 224))
    rgb_resized = np.array(pil_resized).astype(np.float32) / 255.0
    tensor = preprocess(pil_resized)
    return tensor, rgb_resized, pil_resized

# Collect 2 images per class
samples = []   # (tensor, rgb_np, pil, class_name)
for cls in class_names:
    cls_dir = DATA_ROOT / cls
    imgs = sorted(cls_dir.glob('*.jpg'))[:2]
    for p in imgs:
        t, rgb, pil = load_sample(p)
        samples.append((t, rgb, pil, cls))

print(f'Loaded {len(samples)} sample images across {len(class_names)} classes')

## 3. GradCAM — Gradient-weighted Class Activation Maps

GradCAM (Selvaraju et al., 2017) computes a coarse localisation map by:
1. Running a forward pass and selecting the target class score
2. Computing gradients of that score with respect to the **final convolutional feature map**
3. Global average pooling the gradients → per-channel weights
4. Weighted sum of feature maps → heat map, ReLU'd to keep only positive activations

We target `model.layer4[-1]` — the last residual block of ResNet-50 which captures
the highest-level semantic features (7×7 spatial resolution before global pooling).

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target the last conv layer of ResNet-50
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# Run GradCAM on all samples
gradcam_results = []  # (rgb_np, cam_image, predicted_class, true_class, confidence)

for tensor, rgb_np, pil, true_cls in samples:
    input_tensor = tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        probs  = torch.softmax(logits, dim=1).squeeze()
        conf, pred_idx = probs.max(0)
    predicted_cls = class_names[int(pred_idx)]
    
    # GradCAM for the predicted class
    targets = [ClassifierOutputTarget(int(pred_idx))]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]  # H×W
    cam_image = show_cam_on_image(rgb_np, grayscale_cam, use_rgb=True)
    
    gradcam_results.append((rgb_np, cam_image, predicted_cls, true_cls, float(conf)))

print(f'GradCAM computed for {len(gradcam_results)} images')

In [ ]:
# Visualise GradCAM — 2 columns (original | heatmap) per class
n_classes = len(class_names)
fig, axes = plt.subplots(n_classes, 4, figsize=(16, n_classes * 2.2))
fig.suptitle('GradCAM Heatmaps — What does ResNet-50 look at?', fontsize=14, fontweight='bold', y=1.01)

for row, cls in enumerate(class_names):
    cls_results = [(r, c, p, t, cf) for r, c, p, t, cf in gradcam_results if t == cls]
    for col_pair, (rgb, cam_img, pred, true, conf) in enumerate(cls_results[:2]):
        col_base = col_pair * 2
        axes[row, col_base].imshow(rgb)
        axes[row, col_base].set_title(f'True: {true}', fontsize=7)
        axes[row, col_base].axis('off')
        
        color = 'green' if pred == true else 'red'
        axes[row, col_base + 1].imshow(cam_img)
        axes[row, col_base + 1].set_title(f'Pred: {pred} ({conf:.0%})', fontsize=7, color=color)
        axes[row, col_base + 1].axis('off')

plt.tight_layout()
out_path = FIGURES_DIR / 'gradcam_heatmaps.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

## 4. SHAP DeepExplainer

SHAP (SHapley Additive exPlanations, Lundberg & Lee 2017) assigns each input pixel
a *Shapley value* — how much it contributed to pushing the prediction above or below
the expected baseline. Red pixels = pushed prediction higher; blue = pushed it lower.

**Difference from GradCAM:**
- GradCAM gives a *spatial* map (where the model looks)
- SHAP gives a *signed* map (which pixels increase vs decrease confidence)
- SHAP is theoretically grounded in cooperative game theory; GradCAM is faster

For poverty estimation, SHAP is particularly useful to audit whether the model
is penalising or rewarding specific infrastructure signatures.

In [ ]:
import shap

# Background dataset: 50 random images for SHAP baseline
all_tensors = torch.stack([t for t, _, _, _ in samples])  # (N, 3, 224, 224)
background = all_tensors[:min(20, len(all_tensors))].to(device)

# Use 5 test images (one per distinct class for variety)
seen_classes = set()
test_samples = []
for t, rgb, pil, cls in samples:
    if cls not in seen_classes:
        test_samples.append((t, rgb, cls))
        seen_classes.add(cls)
    if len(test_samples) == 5:
        break

test_tensors = torch.stack([t for t, _, _ in test_samples]).to(device)

print('Computing SHAP values (this takes ~30–60 seconds)...')
explainer   = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(test_tensors)  # list of arrays, one per class
print('Done.')

In [ ]:
# For each test image, plot SHAP values for its predicted class
fig, axes = plt.subplots(5, 3, figsize=(12, 18))
fig.suptitle('SHAP DeepExplainer — Pixel Contributions to Prediction', fontsize=13, fontweight='bold')

for i, (tensor, rgb_np, true_cls) in enumerate(test_samples):
    with torch.no_grad():
        probs = torch.softmax(model(tensor.unsqueeze(0).to(device)), dim=1).squeeze()
        pred_idx = int(probs.argmax())
        conf     = float(probs[pred_idx])
    pred_cls = class_names[pred_idx]
    
    # SHAP values for the predicted class: shape (3, 224, 224) → aggregate over channels
    sv = shap_values[pred_idx][i]          # (3, 224, 224)
    sv_rgb = np.transpose(sv, (1, 2, 0))   # (224, 224, 3)
    sv_abs = np.abs(sv_rgb).mean(axis=2)   # (224, 224) — magnitude map
    sv_sum = sv_rgb.sum(axis=2)            # (224, 224) — signed sum
    
    axes[i, 0].imshow(rgb_np)
    axes[i, 0].set_title(f'Input\nTrue: {true_cls}', fontsize=8)
    axes[i, 0].axis('off')
    
    im1 = axes[i, 1].imshow(sv_abs, cmap='hot')
    axes[i, 1].set_title(f'SHAP magnitude\nPred: {pred_cls} ({conf:.0%})', fontsize=8)
    axes[i, 1].axis('off')
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046)
    
    vmax = np.percentile(np.abs(sv_sum), 99)
    im2 = axes[i, 2].imshow(sv_sum, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[i, 2].set_title('SHAP signed\n(red=+confidence, blue=−confidence)', fontsize=8)
    axes[i, 2].axis('off')
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

plt.tight_layout()
out_path = FIGURES_DIR / 'shap_explanations.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

## 5. Research Implications

### What we observe
- GradCAM concentrates on **texture and structural patterns**: road edges for Highway,
  canopy texture for Forest, building grid patterns for Residential and Industrial.
- SHAP signed maps reveal that **bright, high-reflectance pixels push confidence up**
  for Industrial (rooftops, concrete) while **dark, low-reflectance pixels push up**
  confidence for Forest (dense canopy).

### Direct connection to poverty estimation
The visual features our EuroSAT model attends to are the *same types* of features
used in poverty prediction research:

| Visual feature | EuroSAT class where it fires | Poverty relevance |
|---|---|---|
| Regular building grids | Residential / Industrial | Formal housing = higher income |
| Dense irregular structures | Residential edge cases | Informal settlements = lower income |
| Road presence | Highway | Infrastructure access = wealth proxy |
| Bare soil | AnnualCrop / Pasture | Subsistence farming = rural poverty |
| Vegetation density | Forest / HerbaceousVegetation | Food security indicator |

### Next step for poverty research
Fine-tune this model on a **wealth-labelled satellite dataset** (e.g. DHS survey
clusters matched to Sentinel-2 tiles as in Yeh et al. 2020) and apply the same
GradCAM pipeline — the heatmaps would then show which visual features the model
uses as poverty proxies, enabling bias auditing and policy-relevant interpretation.

### References
- Selvaraju et al. (2017). *GradCAM: Visual Explanations from Deep Networks*. ICCV.
- Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
- Jean et al. (2016). *Combining satellite imagery and ML to predict poverty*. Science.
- Yeh et al. (2020). *Using publicly available satellite imagery for poverty mapping*. Nature Communications.